# RAG Notebook 6 — Indic-GPT (GPT-2 trained on Kannada corpus)

**Model:** `aashay96/indic-gpt` — GPT-2 fine-tuned on the AI4Bharat Indic corpus
which includes Kannada. ~117M parameters. Very fast, fully public, no gate.

**Why use this model:** It represents the lightest possible Kannada-trained generative model.
Comparing it against Sarvam-2B (2B) and Sarvam-1 (2B) shows how much model size
matters for Kannada agricultural QA.

**Pipeline:** Question → FAISS + Cross-encoder reranking (top-3) → Indic-GPT generates answer

**Metrics:** BLEU · BERTScore F1 · Answer Similarity · Context Relevance · Faithfulness

**No HuggingFace login needed.**

## Files needed
```
/content/rag_eval_set.json
/content/drive/MyDrive/AgriQADataset/AgriQADataset/Central Govt Schemes/
/content/drive/MyDrive/AgriQADataset/AgriQADataset/Karnataka State Schemes/
```
**Runtime:** T4 GPU (or even CPU — model is tiny)

## ── CELL 1 ── Install packages

In [2]:
!pip install -q transformers==4.46.0 sentencepiece accelerate
!pip install -q faiss-cpu sentence-transformers
!pip install -q sacrebleu bert-score
print('✅ Packages installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00
✅ Packages installed


## ── CELL 2 ── Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


## ── CELL 3 ── Paths and config

In [3]:
import os, json, re, glob
import numpy as np
import torch

EVAL_JSON     = '/content/rag_eval_set.json'
CENTRAL_DIR   = '/content/drive/MyDrive/AgriQADataset/AgriQADataset/Central Govt Schemes'
KARNATAKA_DIR = '/content/drive/MyDrive/AgriQADataset/AgriQADataset/Karnataka State Schemes'
RESULTS_DIR   = '/content/drive/MyDrive/AgriQA/rag_comparison_results'

RETRIEVER_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
RERANKER_MODEL  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
GEN_MODEL       = 'aashay96/indic-gpt'   # GPT-2 on Indic corpus, includes Kannada

CHUNK_SIZE    = 400
CHUNK_OVERLAP = 80
TOP_K         = 3
RERANK_POOL   = 10

os.makedirs(RESULTS_DIR, exist_ok=True)
assert os.path.exists(EVAL_JSON),     f'Not found: {EVAL_JSON}'
assert os.path.exists(CENTRAL_DIR),   f'Not found: {CENTRAL_DIR}'
assert os.path.exists(KARNATAKA_DIR), f'Not found: {KARNATAKA_DIR}'
print('✅ Paths verified')

✅ Paths verified


## ── CELL 4 ── GPU check

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device : cpu


## ── CELL 5 ── Load and chunk scheme files

In [5]:
def read_scheme(path):
    with open(path, encoding='utf-8') as f:
        raw = f.read().replace('\r\n','\n').replace('\r','\n')
    txt = re.sub(r'"([^"]{2,40})":', r'\1:', raw)
    txt = re.sub(r'["\[\]{}]', ' ', txt)
    txt = re.sub(r',\s*\n', '\n', txt)
    txt = re.sub(r'[ \t]+', ' ', txt)
    txt = re.sub(r'\n{3,}', '\n\n', txt)
    return txt.strip()

def chunk_text(text, scheme_no):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        if end < len(text):
            nb = text.rfind('\n', start, end)
            if nb > start + CHUNK_SIZE // 2:
                end = nb
        chunk = text[start:end].strip()
        if len(chunk) > 30:
            chunks.append({'scheme_no': scheme_no,
                           'chunk_id': f'{scheme_no}_{len(chunks)}',
                           'text': chunk})
        start = end - CHUNK_OVERLAP if end < len(text) else len(text)
    return chunks

file_map = {}
for d in [CENTRAL_DIR, KARNATAKA_DIR]:
    for p in glob.glob(os.path.join(d, '*.txt')):
        b = os.path.basename(p).replace('.txt','')
        if b.isdigit():
            file_map[int(b)] = p

all_chunks, full_ctx_map = [], {}
for sno in sorted(file_map):
    txt = read_scheme(file_map[sno])
    full_ctx_map[sno] = txt
    all_chunks.extend(chunk_text(txt, sno))

print(f'✅ {len(file_map)} scheme files → {len(all_chunks)} chunks')

✅ 80 scheme files → 720 chunks


## ── CELL 6 ── FAISS index + cross-encoder reranker

In [6]:
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print('Loading retriever …')
retriever = SentenceTransformer(RETRIEVER_MODEL)
embeddings = retriever.encode(
    [c['text'] for c in all_chunks],
    batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
print(f'✅ FAISS index — {index.ntotal} vectors')

print('Loading cross-encoder reranker …')
rerank_tok   = AutoTokenizer.from_pretrained(RERANKER_MODEL)
rerank_model = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL)
rerank_model.eval().to(device)
print('✅ Reranker loaded')

def retrieve(question, top_k=TOP_K, pool=RERANK_POOL):
    q = retriever.encode([question], normalize_embeddings=True,
                         convert_to_numpy=True).astype(np.float32)
    scores, idxs = index.search(q, pool)
    candidates = []
    for sc, idx in zip(scores[0], idxs[0]):
        c = dict(all_chunks[idx])
        c['faiss_score'] = float(sc)
        candidates.append(c)
    pairs = [[question, c['text']] for c in candidates]
    enc   = rerank_tok(pairs, padding=True, truncation=True,
                       max_length=512, return_tensors='pt').to(device)
    with torch.no_grad():
        rerank_scores = rerank_model(**enc).logits.squeeze(-1).cpu().numpy()
    for c, sc in zip(candidates, rerank_scores):
        c['score'] = float(sc)
    candidates.sort(key=lambda x: x['score'], reverse=True)
    return candidates[:top_k]

test = retrieve('ಬೆಳೆ ವಿಮೆ ಯೋಜನೆ')
print(f'Test: scheme={test[0]["scheme_no"]} score={test[0]["score"]:.3f}')

Loading retriever …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

✅ Reranker loaded
Test: scheme=79 score=7.814


## ── CELL 7 ── Load evaluation set

In [7]:
with open(EVAL_JSON, encoding='utf-8') as f:
    eval_set = json.load(f)
print(f'✅ {len(eval_set)} evaluation questions')

✅ 15 evaluation questions


## ── CELL 8 ── Define 5 metrics

In [8]:
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn

_bleu = BLEU(effective_order=True)

def compute_all_metrics(predictions, references, questions, contexts):
    bleu_scores = [_bleu.sentence_score(p, [r]).score
                   for p, r in zip(predictions, references)]
    _, _, F1 = bert_score_fn(predictions, references, lang='kn', verbose=False)
    bert_scores = F1.numpy().tolist()
    pred_emb = retriever.encode(predictions, normalize_embeddings=True, convert_to_numpy=True)
    ref_emb  = retriever.encode(references,  normalize_embeddings=True, convert_to_numpy=True)
    q_emb    = retriever.encode(questions,   normalize_embeddings=True, convert_to_numpy=True)
    ctx_emb  = retriever.encode(contexts,    normalize_embeddings=True, convert_to_numpy=True)
    ans_sim  = (pred_emb * ref_emb).sum(axis=1).tolist()
    ctx_rel  = (q_emb   * ctx_emb).sum(axis=1).tolist()
    faithful = (pred_emb * ctx_emb).sum(axis=1).tolist()
    def avg(lst): return round(float(np.mean(lst)), 4)
    def rnd(lst): return [round(float(x), 4) for x in lst]
    return {
        'bleu':              avg(bleu_scores),
        'bertscore_f1':      avg(bert_scores),
        'answer_similarity': avg(ans_sim),
        'context_relevance': avg(ctx_rel),
        'faithfulness':      avg(faithful),
        'bleu_per_q':        rnd(bleu_scores),
        'bert_per_q':        rnd(bert_scores),
        'ans_sim_per_q':     rnd(ans_sim),
        'ctx_rel_per_q':     rnd(ctx_rel),
        'faithful_per_q':    rnd(faithful),
    }

print('✅ 5 metrics defined')

✅ 5 metrics defined


## ── CELL 9 ── Load Indic-GPT

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {GEN_MODEL} …')
gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)

# Set pad token — GPT-2 doesn't have one by default
gen_tok.pad_token = gen_tok.eos_token

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float32,   # tiny model — float32 is fine
)
gen_model.to(device)
gen_model.eval()

param_count = sum(p.numel() for p in gen_model.parameters())/1e6
print(f'✅ Indic-GPT loaded — {param_count:.0f}M parameters')

Loading aashay96/indic-gpt …


tokenizer_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

✅ Indic-GPT loaded — 132M parameters


## ── CELL 10 ── Indic-GPT prompt and inference

Indic-GPT is a GPT-2-style completion model.
Given a Kannada context and question, it continues the text.
We use a tight 1-shot prompt to keep it on track.

In [10]:
def indicgpt_answer(question, context):
    """
    Indic-GPT is a GPT-2 completion model trained on AI4Bharat Indic corpus.
    We use 1-shot completion format in Kannada.
    Note: This is a small ~117M model, so output quality will be
    lower than Sarvam-1/2B. It serves as a baseline for model size comparison.
    """
    # Keep context short — GPT-2 has 1024 token limit
    prompt = (
        f'ಸಂದರ್ಭ: {context[:400]}\n'
        f'ಪ್ರಶ್ನೆ: {question}\n'
        'ಉತ್ತರ:'
    )

    inputs = gen_tok(
        prompt, return_tensors='pt',
        truncation=True, max_length=800
    ).to(device)

    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=gen_tok.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    answer = gen_tok.decode(new_tokens, skip_special_tokens=True).strip()
    # Stop at next section markers
    for stop in ['\nಸಂದರ್ಭ:', '\nಪ್ರಶ್ನೆ:', '\n\n']:
        if stop in answer:
            answer = answer.split(stop)[0].strip()
    return answer if answer else 'ಈ ಮಾಹಿತಿ ಲಭ್ಯವಿಲ್ಲ'

# Quick test
test_hits = retrieve('ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಉದ್ದೇಶ ಏನು?')
test_ctx  = '\n\n'.join(h['text'] for h in test_hits)
test_ans  = indicgpt_answer('ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಉದ್ದೇಶ ಏನು?', test_ctx)
print('Test answer:', test_ans[:200])

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Test answer: ಸೂಡಿ, ನೀಕರಣ, ಶಾಂತದೊಂಡಿಯಲ್ತವಾರರು, ಭಾಲ್ಪಟ್ಯವಾನವೆಂದಾರಿಯಾರರುವಾಗಿಯಾಗಿಯಾಗಿಯಾಗಿಯಾಗೆಯಾಗಿಯಾಗಿಯಾಗಿಯಾಗಿಯಾಗೆಯಾ


## ── CELL 11 ── Run RAG on all 15 questions

In [11]:
predictions, references, questions_list, contexts_list = [], [], [], []
rows = []

for qa in eval_set:
    hits    = retrieve(qa['question'])
    context = '\n\n'.join(h['text'] for h in hits)
    top_ctx = hits[0]['text']
    answer  = indicgpt_answer(qa['question'], context)

    predictions.append(answer)
    references.append(qa['answer'])
    questions_list.append(qa['question'])
    contexts_list.append(top_ctx)

    rows.append({
        'qid':               qa['qid'],
        'scheme_no':         qa['scheme_no'],
        'question':          qa['question'],
        'gold':              qa['answer'],
        'predicted':         answer,
        'top_scheme_hit':    hits[0]['scheme_no'],
        'retrieval_correct': qa['scheme_no'] == hits[0]['scheme_no'],
        'rerank_score':      round(hits[0]['score'], 4),
    })
    print(f'{qa["qid"]} ✓  →  {answer[:80]}')

print(f'\n✅ Inference done — {len(predictions)} answers')

E01 ✓  →  ಸೂಡಿ, ನೀಕರಣ, ಶಾಂತದೊಂಡಿಯಲ್ತವಾರರು, ಭಾಲ್ಪಟ್ಯವಾನವೆಂದಾರಿಯಾರರುವಾಗಿಯಾಗಿಯಾಗಿಯಾಗಿಯಾಗೆಯಾಗಿ
E02 ✓  →  ಪ್ತವಾರರ ಮುಂಬಾಗಿಯಲ್ಪಟ್ಪಡಿಯಿಂದು, ಶಾಂಗದ ಜಾಗಿಯಾಗೆಯಾಗೆಯಾಗೆಯಾಗೆಯಾಗೆಯಾಗಿಯಾಗಿಯಾಗೆಯೆಂದಾಗಿ
E03 ✓  →  ವಿಗಳು, ಶಿಯಲ್ಪಟ್ಯದಿಂದಾರರೊಂಡಿ, ನಿಯಾಧಿಯಾಶಿಯಾಳಿಯಾರಾರಾರಾನತೆ, ಹಾಜ್ತನಾಗಿಯಾರ, ಗ್ತ್ತ್ತ್ತ್
E04 ✓  →  ದಾರರೊಂಡರು, ಶಿಯಲ್ತ್ತನಾಶಿಯಾಳೆ, ಪರಿಯನ್ತ್ತ್ಯವಾಮಾವನ್ತ್ಕ್ತ್ಪಟ್ತತೆಯಾರಾರ್ತ್ತ್ತ್ತ್ತವಾಗಿಯಾ
E05 ✓  →  ಭಾಧಿಗಳು, ಶಾಂಡಿ, ಜಾರರಾಗಿಯಲ್ಯವಾಜ್ಪಟ್ತದ ಪರಿಯಾಗಿಯಾಯಾರಿಯಾಯಕ್ತನ, ಗ್ತತೆ, ತಾರಾಗೆಯಾವನ್ತ್ಕ
E06 ✓  →  ಹಾಧಿಯಲ್ತ್ತ್ವಿಯಾರರು, ಭಾರಾರಾರಿಯಾನ್ಕ್ತ್ಪಟ್ತ್ತ್ಕ್ತ್ತ್ಕ್ಪಡಿಯಾರ್ತ್ತ್ಕರ್ತ್ತ್ಕ್ಕ್ತನಾನದೊಂ
E07 ✓  →  ವಿಗಳು, ಮತ್ಯವಾರರೊಂಡರೆ, ಭಾರಾಗೆಯನ್ಪಟ್ತದಿಂದಾಗೆಯಾಗೆಯಾಗೆಯಾಗೆಯಾಗೆಗಳನ್ಪಡಿಯಾಗಿಯಾಗೆಯಾರಾಗಿಯ
E08 ✓  →  ಪ್ಯವಾರರೊಂಡಿ, ಗ್ಪಟ್ಯವನ್ಯದಾಮಾರಿಯಾಗೆಯನ್ಗದ, ಶಾಂಗಣಿಯೆಂಬಿಯಾಗೆಯಾಗೆಯಾರಾಗೆಯಾಗಿಯಾಗಿಯಾಗೆಯಾಗ
E09 ✓  →  ಪ್ತವಾರರು, ವಿಗಳಿಂದಾಲ್ಯವಾಡಿಯಾಮಾಗಿಯಲ್ತತೆಯನ್ಪಟ್ತದ ಪರಿಯಾಗಿಯಾರ, ಶಾಂತಕ್ತನಾಗೆಯಾನತೆಯಾವನ್ಯ
E10 ✓  →  ಭಾಗಿಯಲ್ತತೆ, ಶಾಂಗಣಿಯಿಂದು, ಜಾರರೊಂಡರು. ಈ ಸಂತಹ, ತುಂಬ್ತನಾಶಿಯಾರ್ಯವಾನದಿಂದಾಗಿಯಾಗಿಯಾಗಿ, ಪ
E11 ✓  →  ನಿಯಲ್ಪಟ್ಕ್ಯವಾಶಿಯಿಂದಾರರೊಂಡಿ, ಭಾರ, ಶಾಂಗದ ಜಾಗಿಯಾವನ್ಕ್ಕ್ಕ್ತ್ಯದಾರಿಯಾನತೆ, ಬೆಂಬ್ಕ್ಕ್ಕ್ತ

## ── CELL 12 ── Compute all 5 metrics

In [12]:
print('Computing metrics …')
metrics = compute_all_metrics(predictions, references, questions_list, contexts_list)

print()
print('=' * 55)
print('  Indic-GPT RAG — RESULTS')
print('=' * 55)
print(f'  BLEU                        : {metrics["bleu"]}')
print(f'  BERTScore F1                : {metrics["bertscore_f1"]}')
print(f'  Answer Similarity  (RAGAS)  : {metrics["answer_similarity"]}')
print(f'  Context Relevance  (RAGAS)  : {metrics["context_relevance"]}')
print(f'  Faithfulness       (RAGAS)  : {metrics["faithfulness"]}')
print('=' * 55)

Computing metrics …


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


  Indic-GPT RAG — RESULTS
  BLEU                        : 0.4543
  BERTScore F1                : 0.5972
  Answer Similarity  (RAGAS)  : 0.7128
  Context Relevance  (RAGAS)  : 0.8104
  Faithfulness       (RAGAS)  : 0.7175


## ── CELL 13 ── Per-question breakdown

In [13]:
import pandas as pd

df = pd.DataFrame(rows)
df['bleu']     = metrics['bleu_per_q']
df['bert']     = metrics['bert_per_q']
df['ans_sim']  = metrics['ans_sim_per_q']
df['ctx_rel']  = metrics['ctx_rel_per_q']
df['faithful'] = metrics['faithful_per_q']

print('Per-question scores:')
print(df[['qid','bleu','bert','ans_sim','ctx_rel','faithful','retrieval_correct']].to_string(index=False))
print(f'\nRetrieval accuracy: {df["retrieval_correct"].mean()*100:.1f}%')

Per-question scores:
qid   bleu   bert  ans_sim  ctx_rel  faithful  retrieval_correct
E01 0.0000 0.6039   0.7812   0.7540    0.8348              False
E02 0.0879 0.6248   0.8297   0.8136    0.8693              False
E03 0.0000 0.5800   0.3341   0.9037    0.8713              False
E04 0.0000 0.5757   0.5930   0.8829    0.7992              False
E05 0.0000 0.6095   0.8356   0.6550    0.4840               True
E06 0.0000 0.5551   0.0480   0.6547    0.2435              False
E07 1.4457 0.5980   0.9260   0.8966    0.9076              False
E08 0.0000 0.6049   0.7743   0.8785    0.8499              False
E09 0.0000 0.5645   0.8567   0.8184    0.7873              False
E10 2.6374 0.6360   0.4483   0.8946    0.4537              False
E11 0.9836 0.6303   0.9082   0.6567    0.7153              False
E12 0.0000 0.5690   0.7717   0.9289    0.8347              False
E13 0.0000 0.6175   0.8285   0.6000    0.3802              False
E14 1.6605 0.5955   0.9261   0.9078    0.8744              False
E15 

## ── CELL 14 ── Save results

In [14]:
csv_path = f'{RESULTS_DIR}/indicgpt_rag_results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ CSV → {csv_path}')

result_dict = {
    'model': GEN_MODEL,
    'type': 'generative_gpt2_indic',
    'prompt_format': '1-shot completion ಸಂದರ್ಭ/ಪ್ರಶ್ನೆ/ಉತ್ತರ',
    'reranking': True,
    'eval_size': len(eval_set),
    'bleu':              metrics['bleu'],
    'bertscore_f1':      metrics['bertscore_f1'],
    'answer_similarity': metrics['answer_similarity'],
    'context_relevance': metrics['context_relevance'],
    'faithfulness':      metrics['faithfulness'],
}
json_path = f'{RESULTS_DIR}/indicgpt_rag_metrics.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(result_dict, f, indent=2, ensure_ascii=False)
print(f'✅ Metrics JSON → {json_path}')

✅ CSV → /content/drive/MyDrive/AgriQA/rag_comparison_results/indicgpt_rag_results.csv
✅ Metrics JSON → /content/drive/MyDrive/AgriQA/rag_comparison_results/indicgpt_rag_metrics.json
